In [ ]:
import json
import os

import requests

In [ ]:
os.environ["BALANCE_API_BASE_URL"] = (
    "https://microsoft-foundry-playground-balance-mock-api.9uxguk.easypanel.host"
)

In [ ]:
os.environ.get("BALANCE_API_BASE_URL", "http://localhost:8000")

In [ ]:
def get_base_url() -> str:
    return os.environ.get("BALANCE_API_BASE_URL", "http://localhost:8000")


def call_balance_api(customer_id: str) -> dict:
    url = f"{get_base_url().rstrip('/')}/api/balance"
    payload = {"customer_id": customer_id}
    response = requests.post(url, json=payload, timeout=5)
    try:
        data = response.json()
    except json.JSONDecodeError:
        data = {"raw_text": response.text}
    return {
        "status_code": response.status_code,
        "data": data,
    }

In [ ]:
result = call_balance_api("customer-002")
print(json.dumps(result, indent=2, ensure_ascii=False))

---
## Tokens Test

In [ ]:
import os

import dotenv

# import requests

In [ ]:
dotenv.load_dotenv()

In [ ]:
KEYCLOAK_BASE_URL = os.getenv("KEYCLOACK_BASE_URL")
REALM_NAME = os.getenv("REALM_NAME")
CLIENT_ID = os.getenv("CLIENT_ID")
CLIENT_SECRET = os.getenv("CLIENT_SECRET")
USER_1_NAME = os.getenv("USER_1_NAME")
USER_2_NAME = os.getenv("USER_2_NAME")
USER_1_PASSWORD = os.getenv("USER_1_PASSWORD")
USER_2_PASSWORD = os.getenv("USER_2_PASSWORD")

TOKEN_URL = f"{KEYCLOAK_BASE_URL}/realms/{REALM_NAME}/protocol/openid-connect/token"

In [ ]:
def get_access_token(username: str, password: str):
    data = {
        "grant_type": "password",
        "client_id": CLIENT_ID,
        "client_secret": CLIENT_SECRET,
        "username": username,
        "password": password,
    }

    headers = {"Content-Type": "application/x-www-form-urlencoded"}

    response = requests.post(TOKEN_URL, data=data, headers=headers)

    if response.status_code != 200:
        raise Exception(f"Error {response.status_code}: {response.text}")

    return response.json()["access_token"]

In [ ]:
token = get_access_token(USER_1_NAME, USER_1_PASSWORD)
print(token)

In [ ]:
token = get_access_token(USER_2_NAME, USER_2_PASSWORD)
print(token)

### Use Token to Call API

In [ ]:
response = requests.post(
    "http://localhost:8000/api/balance",
    headers={
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json",
    },
    json={},
    timeout=10,
)

print(response.status_code)
print(response.text)

In [ ]:
response = requests.post("http://localhost:8000/api/balance", json={})

assert str(response) == '<Response [401]>'

---
## Chatbot API

In [ ]:
response = requests.post(
    "http://localhost:5000/chat", json={"message": "hola, quiero ver mi balance"}
)

In [ ]:
print(response.status_code)
print(response.text)

---
## Ver Mensajes Agente

In [ ]:
import json
import os

import dotenv
import requests
from azure.ai.agents import AgentsClient

# from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

In [ ]:
dotenv.load_dotenv()

In [ ]:
PROJECT_ENDPOINT = os.getenv("PROJECT_ENDPOINT")
BALANCE_AGENT_ID = os.getenv("BALANCE_AGENT_ID")

In [ ]:
client = AgentsClient(endpoint=PROJECT_ENDPOINT, credential=DefaultAzureCredential())

In [ ]:
thread_id = 'thread_GuZNYIaF1uq5d8IT0ePpIfMd'
run_id = 'run_0111CChWZVZiS2EAOIGSJ9rr'

In [ ]:
def format_message_content(content):
    return json.dumps(
        json.loads(str(content).replace("'", '"')), indent=2, ensure_ascii=False
    )

In [ ]:
last_5_messages = client.messages.list(thread_id=thread_id, limit=10)
print("Last 10 messages in the thread:")

for msg in last_5_messages:
    print(f"\n{msg.role}: {format_message_content(msg.content)}")

In [ ]:
run = client.runs.get(thread_id=thread_id, run_id=run_id)

In [ ]:
dict(run)

In [ ]:
stream = client.runs.stream(thread_id=thread_id, agent_id=BALANCE_AGENT_ID)

In [ ]:
type(stream)

In [ ]:
# with client.runs.stream(thread_id=thread_id, agent_id=BALANCE_AGENT_ID) as stream:
#     for event_type, event_data, _ in stream:
#         print(f"\nEVENTO: {event_type}")

#         if isinstance(event_data, MessageDeltaChunk):
#             print("DELTA:", event_data.text)

#         elif isinstance(event_data, ThreadMessage):
#             print("MENSAJE:")
#             print("  id:", event_data.id)
#             print("  role:", event_data.role)
#             print("  status:", event_data.status)

#         elif isinstance(event_data, ThreadRun):
#             print("RUN:")
#             print("  id:", event_data.id)
#             print("  status:", event_data.status)
#             if getattr(event_data, "last_error", None):
#                 print("  last_error:", event_data.last_error)

#             if getattr(event_data, "required_action", None):
#                 print("  required_action:", event_data.required_action)

#         elif isinstance(event_data, RunStep):
#             print("STEP:")
#             print("  id:", event_data.id)
#             print("  type:", event_data.type)
#             print("  status:", event_data.status)
#             if getattr(event_data, "last_error", None):
#                 print("  last_error:", event_data.last_error)

#         elif event_type == AgentStreamEvent.ERROR:
#             print("ERROR STREAM:", event_data)

#         elif event_type == AgentStreamEvent.DONE:
#             print("STREAM TERMINADO")
#             break

#         else:
#             print("DATA:", event_data)